# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
# Import pandas
import pandas as pd

# Load the three datasets
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# Combine the datasets vertically
df = pd.concat([df1, df2, df3], ignore_index=True)

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Rename inconsistent columns
df = df.rename(columns={
    "st": "state",
    "employmentstatus": "employment_status"
})

# Standardize gender values
df["gender"] = df["gender"].replace({
    "M": "Male",
    "F": "Female",
    "Femal": "Female"
})

# Standardize state names
df["state"] = df["state"].replace({
    "AZ": "Arizona",
    "Cali": "California",
    "WA": "Washington"
})

# Standardize education values
df["education"] = df["education"].replace({
    "Bachelors": "Bachelor"
})

# Remove percentage signs and convert customer lifetime value to numeric
df["customer_lifetime_value"] = (
    df["customer_lifetime_value"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

df["customer_lifetime_value"] = pd.to_numeric(
    df["customer_lifetime_value"],
    errors="coerce"
)

# Extract the number of open complaints
df["number_of_open_complaints"] = (
    df["number_of_open_complaints"]
    .astype(str)
    .str.split("/")
    .str[1]
)

df["number_of_open_complaints"] = pd.to_numeric(
    df["number_of_open_complaints"],
    errors="coerce"
)

# Combine similar vehicle classes
df["vehicle_class"] = df["vehicle_class"].replace({
    "Sports Car": "Luxury",
    "Luxury SUV": "Luxury",
    "Luxury Car": "Luxury"
})

# find actual date colomn
print(df.columns.tolist()
)

# Remove completely empty rows and duplicate rows
df = df.dropna(how="all")
df = df.drop_duplicates()

# Reset the index
df = df.reset_index(drop=True)

# Display the cleaned combined dataset
print(df.shape)
df.head()

['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount', 'state', 'gender']
(9134, 13)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount,state,gender
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,0.0,Personal Auto,Four-Door Car,2.704934,NaN,NaN
1,QZ44356,Arizona,Female,Bachelor,697953.59,0.0,94.0,0.0,Personal Auto,Four-Door Car,1131.464935,NaN,NaN
2,AI49188,Nevada,Female,Bachelor,1288743.17,48767.0,108.0,0.0,Personal Auto,Two-Door Car,566.472247,NaN,NaN
3,WW63253,California,Male,Bachelor,764586.18,0.0,106.0,0.0,Corporate Auto,SUV,529.881344,NaN,NaN
4,GA49547,Washington,Male,High School or Below,536307.65,36357.0,68.0,0.0,Personal Auto,Four-Door Car,17.269323,NaN,NaN


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [3]:
# Import pandas
import pandas as pd

# Load the cleaned marketing customer dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
marketing_df = pd.read_csv(url)

# Standardize column names
marketing_df.columns = (
    marketing_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Check the available columns
print("Columns:")
print(marketing_df.columns.tolist())


# ---------------------------------------------------
# QUESTION 1
# Total revenue per sales channel
# ---------------------------------------------------

sales_channel_revenue = pd.pivot_table(
    marketing_df,
    index="sales_channel",
    values="customer_lifetime_value",
    aggfunc="sum"
).round(2)

# Sort from highest to lowest revenue
sales_channel_revenue = sales_channel_revenue.sort_values(
    by="customer_lifetime_value",
    ascending=False
)

print("\nTotal revenue per sales channel:")
display(sales_channel_revenue)

# Identify the highest-performing sales channel
top_channel = sales_channel_revenue.index[0]
top_revenue = sales_channel_revenue.iloc[0, 0]

print(
    f"\nInsight: {top_channel} generated the highest total revenue "
    f"with {top_revenue:,.2f}."
)


# ---------------------------------------------------
# QUESTION 2
# Average customer lifetime value by gender and education
# ---------------------------------------------------

average_clv = pd.pivot_table(
    marketing_df,
    index="education",
    columns="gender",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

print("\nAverage customer lifetime value by gender and education:")
display(average_clv)

# Find the highest average CLV combination
highest_combination = average_clv.stack().idxmax()
highest_value = average_clv.stack().max()

print(
    f"\nInsight: The highest average customer lifetime value belongs to "
    f"{highest_combination[1]} customers with {highest_combination[0]} education "
    f"at {highest_value:,.2f}."
)


# ---------------------------------------------------
# BONUS
# Number of complaints by policy type and month
# ---------------------------------------------------

# Convert date column to datetime
marketing_df["effective_to_date"] = pd.to_datetime(
    marketing_df["effective_to_date"],
    errors="coerce"
)

# Extract month name
marketing_df["month"] = marketing_df["effective_to_date"].dt.month_name()

# Create summary table
complaints_table = pd.pivot_table(
    marketing_df,
    index="policy_type",
    columns="month",
    values="number_of_open_complaints",
    aggfunc="sum",
    fill_value=0
)

# Convert the table from wide format to long format
complaints_long = (
    complaints_table
    .reset_index()
    .melt(
        id_vars="policy_type",
        var_name="month",
        value_name="number_of_complaints"
    )
    .sort_values(
        by=["policy_type", "number_of_complaints"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("\nComplaints by policy type and month in long format:")
display(complaints_long)

# Show the month with the most complaints for each policy type
highest_complaint_month = complaints_long.loc[
    complaints_long.groupby("policy_type")["number_of_complaints"].idxmax()
]

print("\nMonths with the highest complaints per policy type:")
display(highest_complaint_month)

Columns:
['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']

Total revenue per sales channel:


,customer_lifetime_value
sales_channel,
Agent,33057887.85
Branch,24359201.21
Call Center,17364288.37
Web,12697632.90



Insight: Agent generated the highest total revenue with 33,057,887.85.

Average customer lifetime value by gender and education:


gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83



Insight: The highest average customer lifetime value belongs to F customers with High School or Below education at 8,675.22.

Complaints by policy type and month in long format:


,policy_type,month,number_of_complaints
0,Corporate Auto,January,443.434952
1,Corporate Auto,February,385.208135
2,Personal Auto,January,1727.605722
3,Personal Auto,February,1453.684441
4,Special Auto,February,95.226817
5,Special Auto,January,87.074049



Months with the highest complaints per policy type:


,policy_type,month,number_of_complaints
0,Corporate Auto,January,443.434952
2,Personal Auto,January,1727.605722
4,Special Auto,February,95.226817


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here